In [3]:
# ============================================================
# TRUSTSYN CATBOOST V2 STACKING TABLE GENERATOR
# RANDOM / COLD_COMBINATION / COLD_CELL
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor

# -----------------------------
# PATHS
# -----------------------------

ROOT = Path("/Users/konuri/stacking")

PACKAGE = ROOT / "CATBOOST_TEAM_PACKAGE"

MODEL_DIR = PACKAGE / "02_Final_Model"

OUT_DIR = PACKAGE


# -----------------------------
# FEATURE FILES
# -----------------------------

MASTER = Path(
    "/Users/konuri/model_final/data/almanac_final_59cell_104drug.csv"
)

CELL = Path(
    "/Users/konuri/TrustSyn/Generated_features/cellminer_features/cellminer_rna_features_59cell.csv"
)

STRING = Path(
    "/Users/konuri/TrustSyn/Generated_features/network_features/drug_pair_string_features_final_master_aligned.csv"
)

KEGG = Path(
    "/Users/konuri/TrustSyn/Generated_features/network_features/drug_pair_kegg_features_104drug_master_aligned_canonical_pairs.csv"
)

TARGET = Path(
    "/Users/konuri/TrustSyn/Generated_features/drug_features/drug_pair_target_features_104drug_master_aligned_internal_ids_canonical_pairs.csv"
)

SIM = Path(
    "/Users/konuri/TrustSyn/Generated_features/drug_features/drug_pair_similarity_features_104drug_master_aligned_internal_ids_canonical_pairs.csv"
)


# -----------------------------
# LOAD
# -----------------------------

master = pd.read_csv(MASTER)

cell = pd.read_csv(CELL)

string = pd.read_csv(STRING)

kegg = pd.read_csv(KEGG)

target = pd.read_csv(TARGET)

similarity = pd.read_csv(SIM)


print("MASTER", master.shape)
print("CELL", cell.shape)
print("STRING", string.shape)


# -----------------------------
# CLEAN FEATURE TABLES
# -----------------------------

# STRING already cell-expanded
string = (
    string
    .drop_duplicates(
        subset=["drug_A","drug_B"]
    )
)


# KEGG
kegg = kegg.rename(
    columns={
        "drug_A_id":"drug_A",
        "drug_B_id":"drug_B"
    }
)


# TARGET
target = target.rename(
    columns={
        "drug_A_id":"drug_A",
        "drug_B_id":"drug_B"
    }
)


# SIM
similarity = similarity.rename(
    columns={
        "drug_A_id":"drug_A",
        "drug_B_id":"drug_B",
        "tanimoto_similarity":"Tanimoto_similarity"
    }
)


# -----------------------------
# ADD CELL PCA
# -----------------------------

pc_cols = [
    f"CellMiner_PC{i}"
    for i in range(1,51)
]


# if PCA columns already exist
if len([c for c in cell.columns if "CellMiner_PC" in c]) == 50:

    cell_pc = cell[
        ["cellminer_cellline_id"] +
        pc_cols
    ]

else:

    raise Exception(
        "CellMiner file does not contain PC1-PC50"
    )


# -----------------------------
# BUILD TEST FUNCTION
# -----------------------------

feature_cols = pd.read_csv(
    PACKAGE / "CatBoost_62_feature_list.csv"
)["feature_name"].tolist()


def make_table(split_name):

    print("\nPROCESSING", split_name)


    if split_name=="RANDOM":
        test_path = Path(
            "/Users/konuri/model_final/splits/random/test.csv"
        )

    elif split_name=="COLD_COMBINATION":
        test_path = Path(
            "/Users/konuri/model_final/splits/cold_combination/test.csv"
        )

    elif split_name=="COLD_CELL":
        test_path = Path(
            "/Users/konuri/model_final/splits/cold_cell_line/test.csv"
        )


    test = pd.read_csv(test_path)


    # merge only once
    df = (
        test
        .merge(
            string,
            on=["drug_A","drug_B"],
            how="left"
        )
        .merge(
            kegg,
            on=["drug_A","drug_B"],
            how="left"
        )
        .merge(
            target,
            on=["drug_A","drug_B"],
            how="left"
        )
        .merge(
            similarity,
            on=["drug_A","drug_B"],
            how="left"
        )
        .merge(
            cell_pc,
            on="cellminer_cellline_id",
            how="left"
        )
    )


    df = df.drop_duplicates()


    X = df[feature_cols]


    # fill missing
    X = X.fillna(0)


    preds=[]


    models = sorted(
        (MODEL_DIR / split_name).glob("*.cbm")
    )


    print("Models:",len(models))


    for m in models:

        model = CatBoostRegressor()

        model.load_model(m)

        preds.append(
            model.predict(X)
        )


    df["catboost_prediction"] = np.mean(
        preds,
        axis=0
    )

    df["y_true"] = test["combo_score"].values


    outfile = (
        OUT_DIR /
        f"{split_name}_CatBoost_stacking_table.csv"
    )


    df.to_csv(
        outfile,
        index=False
    )


    print("SAVED:",outfile)
    print(df.shape)


# -----------------------------
# RUN
# -----------------------------

for s in [
    "RANDOM",
    "COLD_COMBINATION",
    "COLD_CELL"
]:
    make_table(s)


print("\nDONE")

MASTER (294073, 9)
CELL (59, 20203)
STRING (294073, 4)


Exception: CellMiner file does not contain PC1-PC50